In [ ]:
# Scenario: AI-Powered Project Tracker (Task-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
# - status → The current state of the task (e.g., "in progress", "completed", "blocked").
# - log → A history of all updates, including who made them and when.

# 2. Workflow (Graph of States)
# Each project update flows through nodes until the task status is refreshed:
# - Input Node
# - Team member submits an update (e.g., "Report draft completed").
# - State updates: task = "Q1 financial report"
# - Processing Node (Groq API)
# - Groq interprets the update and assigns a status:
# - status = "completed"
# - Response Node
# - Assistant confirms the update back to the team:
# - "Task Q1 financial report marked as completed."
# - History Node
# - Logs the update:
# - log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

# 1. State Definition
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    response: str
    log: List[Dict]

# 2. Interpret Update (LLM Node)
def interpret_update(state: ProjectState):
    api_key = os.getenv("NVIDIA_API_KEY")

    prompt = f"""
    You are a project management assistant.

    Task: {state['task']}
    Update: {state['update']}

    Classify the task status into ONE of:
    - completed
    - in progress
    - blocked

    Respond ONLY with the status.
    """

    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta/llama3-8b-instruct",
            "messages": [
                {"role": "system", "content": "You are a strict project status classifier."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0
        }
    )

    result = response.json()
    status = result["choices"][0]["message"]["content"].strip().lower()

    return {"status": status}

# 3. Generate Response
def generate_response(state: ProjectState):
    msg = f"✅ Task '{state['task']}' marked as {state['status']}."
    return {"response": msg}

# 4. Log History
def update_log(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {"log": state["log"] + [entry]}

# 5. Print Output
def show_output(state: ProjectState):
    print("\n📌", state["response"])
    print("\n🗂️ Log Entry:", state["log"][-1])
    return {}

# 6. Build Graph
graph = StateGraph(ProjectState)

graph.add_node("interpret", interpret_update)
graph.add_node("respond", generate_response)
graph.add_node("log", update_log)
graph.add_node("output", show_output)

graph.set_entry_point("interpret")
graph.add_edge("interpret", "respond")
graph.add_edge("respond", "log")
graph.add_edge("log", "output")
graph.add_edge("output", END)

# 7. Run Example
if __name__ == "__main__":
    state = {
        "task": "Q1 financial report",
        "update": "Report draft completed",
        "status": "",
        "response": "",
        "log": []
    }

    app = graph.compile()
    result = app.invoke(state)


📌 ✅ Task 'Q1 financial report' marked as in progress.

🗂️ Log Entry: {'task': 'Q1 financial report', 'update': 'Report draft completed', 'status': 'in progress', 'timestamp': '2026-03-20 11:39:45'}
